In [1]:
# Install Qiskit and required packages
!pip install qiskit qiskit-aer qiskit-ibm-runtime -q

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from kaggle_secrets import UserSecretsClient

print("="*80)
print("KL-DIVERGENCE BUG DETECTION: GHZ vs W vs BELL STATES")
print("="*80)

# Utility functions
def kl_divergence(p, q, epsilon=1e-10):
    """Calculate Kullback-Leibler divergence KL(P||Q)"""
    p = np.array(p) + epsilon
    q = np.array(q) + epsilon
    p = p / np.sum(p)
    q = q / np.sum(q)
    return np.sum(p * np.log(p / q))

def get_probability_distribution(counts, n_qubits):
    """Convert Qiskit counts to probability distribution"""
    total_shots = sum(counts.values())
    n_states = 2**n_qubits
    probs = np.zeros(n_states)
    
    for state_str, count in counts.items():
        state_idx = int(state_str, 2) if isinstance(state_str, str) else state_str
        probs[state_idx] = count / total_shots
    
    return probs

print("\nUtility functions loaded")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 58.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 33.0 MB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 34.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.5/378.5 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 61.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 6.7 MB/s eta 0:00:00
KL-DIVERGENCE BUG DETECTION: GHZ vs W vs BELL STATES

Utility functions loaded


In [2]:
def create_noise_model(error_rate=0.02):
    """Create depolarizing noise model to simulate hardware errors"""
    noise_model = NoiseModel()
    
    # Single-qubit gate errors
    error_1q = depolarizing_error(error_rate, 1)
    noise_model.add_all_qubit_quantum_error(error_1q, ['h', 'x', 'u1', 'u2', 'u3'])
    
    # Two-qubit gate errors (typically higher)
    error_2q = depolarizing_error(error_rate * 2, 2)
    noise_model.add_all_qubit_quantum_error(error_2q, ['cx'])
    
    return noise_model

# Create noise model and simulators
error_rate = 0.02  # 2% error per gate
noise_model = create_noise_model(error_rate)
simulator_ideal = AerSimulator()
simulator_noisy = AerSimulator(noise_model=noise_model)

print(f"Noise model created with {error_rate*100}% error rate")
print("Ideal and noisy simulators ready")

Noise model created with 2.0% error rate
Ideal and noisy simulators ready


In [3]:
def create_ghz_circuit(n_qubits=3):
    """
    Create correct GHZ state circuit
    Result: (|000⟩ + |111⟩)/√2
    """
    qc = QuantumCircuit(n_qubits, n_qubits)
    
    # GHZ state preparation
    qc.h(0)  # Hadamard on first qubit
    for i in range(n_qubits - 1):
        qc.cx(i, i + 1)  # CNOT cascade
    
    # Measure all qubits
    qc.measure(range(n_qubits), range(n_qubits))
    
    return qc

def create_buggy_ghz_circuit(n_qubits=3):
    """
    Create buggy GHZ circuit with extra X gate
    Bug: Extra X gate on qubit 1
    Result: (|010⟩ + |101⟩)/√2
    """
    qc = QuantumCircuit(n_qubits, n_qubits)
    
    # Same GHZ preparation
    qc.h(0)
    for i in range(n_qubits - 1):
        qc.cx(i, i + 1)
    
    # BUG: Extra X gate
    qc.x(1)  # This is the bug!
    
    # Measure all qubits
    qc.measure(range(n_qubits), range(n_qubits))
    
    return qc

# Create GHZ circuits
ghz_correct = create_ghz_circuit(3)
ghz_buggy = create_buggy_ghz_circuit(3)

print("GHZ State Correct Circuit:")
print(ghz_correct.draw(output='text'))
print("\nGHZ State Buggy Circuit:")
print(ghz_buggy.draw(output='text'))

GHZ State Correct Circuit:
     ┌───┐          ┌─┐      
q_0: ┤ H ├──■───────┤M├──────
     └───┘┌─┴─┐     └╥┘┌─┐   
q_1: ─────┤ X ├──■───╫─┤M├───
          └───┘┌─┴─┐ ║ └╥┘┌─┐
q_2: ──────────┤ X ├─╫──╫─┤M├
               └───┘ ║  ║ └╥┘
c: 3/════════════════╩══╩══╩═
                     0  1  2 

GHZ State Buggy Circuit:
     ┌───┐          ┌─┐        
q_0: ┤ H ├──■───────┤M├────────
     └───┘┌─┴─┐     └╥┘┌───┐┌─┐
q_1: ─────┤ X ├──■───╫─┤ X ├┤M├
          └───┘┌─┴─┐ ║ └┬─┬┘└╥┘
q_2: ──────────┤ X ├─╫──┤M├──╫─
               └───┘ ║  └╥┘  ║ 
c: 3/════════════════╩═══╩═══╩═
                     0   2   1 


In [4]:
def create_w_state_circuit(n_qubits=3):
    """
    Create W state circuit for n qubits
    W state: equal superposition of all single-excitation states
    For 3 qubits: (|100⟩ + |010⟩ + |001⟩)/√3
    """
    qc = QuantumCircuit(n_qubits, n_qubits)
    
    # W state preparation using controlled rotations
    # Step 1: Create |100⟩ + √2|0⟩|W_2⟩ superposition
    qc.ry(2 * np.arccos(np.sqrt(2/3)), 0)
    
    # Step 2: If qubit 0 is |0⟩, create equal superposition on remaining qubits
    qc.x(0)  # Flip to use as control
    qc.cry(2 * np.arccos(np.sqrt(1/2)), 0, 1)  # Controlled RY
    qc.x(0)  # Flip back
    
    # Step 3: CNOT to distribute to last qubit
    qc.cx(1, 2)
    
    # Measure all qubits
    qc.measure(range(n_qubits), range(n_qubits))
    
    return qc

def create_buggy_w_state_circuit(n_qubits=3):
    """
    Create buggy W state circuit with extra X gate on qubit 1
    This will flip states: |010⟩ ↔ |000⟩, |110⟩ ↔ |100⟩, |011⟩ ↔ |001⟩
    Result: (|110⟩ + |000⟩ + |011⟩)/√3
    """
    qc = QuantumCircuit(n_qubits, n_qubits)
    
    # Same W state preparation
    qc.ry(2 * np.arccos(np.sqrt(2/3)), 0)
    qc.x(0)
    qc.cry(2 * np.arccos(np.sqrt(1/2)), 0, 1)
    qc.x(0)
    qc.cx(1, 2)
    
    # BUG: Extra X gate on qubit 1
    qc.x(1)
    
    # Measure all qubits
    qc.measure(range(n_qubits), range(n_qubits))
    
    return qc

# Create W state circuits
w_correct = create_w_state_circuit(3)
w_buggy = create_buggy_w_state_circuit(3)

print("\nW State Correct Circuit:")
print(w_correct.draw(output='text'))
print("\nW State Buggy Circuit:")
print(w_buggy.draw(output='text'))


W State Correct Circuit:
     ┌───────────┐┌───┐           ┌───┐┌─┐      
q_0: ┤ Ry(1.231) ├┤ X ├─────■─────┤ X ├┤M├──────
     └───────────┘└───┘┌────┴────┐└───┘└╥┘┌─┐   
q_1: ──────────────────┤ Ry(π/2) ├──■───╫─┤M├───
                       └─────────┘┌─┴─┐ ║ └╥┘┌─┐
q_2: ─────────────────────────────┤ X ├─╫──╫─┤M├
                                  └───┘ ║  ║ └╥┘
c: 3/═══════════════════════════════════╩══╩══╩═
                                        0  1  2 

W State Buggy Circuit:
     ┌───────────┐┌───┐           ┌───┐     ┌─┐   
q_0: ┤ Ry(1.231) ├┤ X ├─────■─────┤ X ├─────┤M├───
     └───────────┘└───┘┌────┴────┐└───┘┌───┐└╥┘┌─┐
q_1: ──────────────────┤ Ry(π/2) ├──■──┤ X ├─╫─┤M├
                       └─────────┘┌─┴─┐└┬─┬┘ ║ └╥┘
q_2: ─────────────────────────────┤ X ├─┤M├──╫──╫─
                                  └───┘ └╥┘  ║  ║ 
c: 3/════════════════════════════════════╩═══╩══╩═
                                         2   0  1 


In [5]:
def create_bell_state_circuit(n_qubits=2):
    """
    Create Bell state circuit (Φ⁺)
    Bell state: (|00⟩ + |11⟩)/√2
    Simplest maximally entangled state
    """
    qc = QuantumCircuit(n_qubits, n_qubits)
    
    # Bell state preparation
    qc.h(0)       # Hadamard on first qubit
    qc.cx(0, 1)   # CNOT to entangle
    
    # Measure all qubits
    qc.measure(range(n_qubits), range(n_qubits))
    
    return qc

def create_buggy_bell_state_circuit(n_qubits=2):
    """
    Create buggy Bell state circuit with extra X gate on qubit 1
    This flips second qubit: |00⟩ → |01⟩, |11⟩ → |10⟩
    Result: (|01⟩ + |10⟩)/√2
    """
    qc = QuantumCircuit(n_qubits, n_qubits)
    
    # Same Bell state preparation
    qc.h(0)
    qc.cx(0, 1)
    
    # BUG: Extra X gate on qubit 1
    qc.x(1)
    
    # Measure all qubits
    qc.measure(range(n_qubits), range(n_qubits))
    
    return qc

# Create Bell state circuits
bell_correct = create_bell_state_circuit(2)
bell_buggy = create_buggy_bell_state_circuit(2)

print("\nBell State Correct Circuit:")
print(bell_correct.draw(output='text'))
print("\nBell State Buggy Circuit:")
print(bell_buggy.draw(output='text'))


Bell State Correct Circuit:
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 

Bell State Buggy Circuit:
     ┌───┐          ┌─┐   
q_0: ┤ H ├──■───────┤M├───
     └───┘┌─┴─┐┌───┐└╥┘┌─┐
q_1: ─────┤ X ├┤ X ├─╫─┤M├
          └───┘└───┘ ║ └╥┘
c: 2/════════════════╩══╩═
                     0  1 


In [6]:
def create_grover_circuit(n_qubits=2):
    """Correct Grover's search for |11> (2 qubits, 1 iteration optimal)"""
    qc = QuantumCircuit(n_qubits, n_qubits)
    
    # Equal superposition
    for i in range(n_qubits):
        qc.h(i)
    
    # Oracle: mark |11> (phase flip)
    qc.x(0); qc.x(1)
    qc.cz(0, 1)
    qc.x(1); qc.x(0)
    
    # Diffusion operator
    for i in range(n_qubits):
        qc.h(i)
    qc.x(0); qc.x(1)
    qc.cz(0, 1)
    qc.x(1); qc.x(0)  # ← Standard unflip both
    for i in range(n_qubits):
        qc.h(i)
    
    qc.measure(range(n_qubits), range(n_qubits))
    return qc

def create_buggy_grover_circuit(n_qubits=2):
    """CORRECTED Buggy Grover: extra X gate on qubit 1 (2nd qubit) in diffusion"""
    qc = QuantumCircuit(n_qubits, n_qubits)
    
    # Same oracle
    for i in range(n_qubits):
        qc.h(i)
    qc.x(0); qc.x(1)
    qc.cz(0, 1)
    qc.x(1); qc.x(0)
    
    # Diffusion operator WITH EXTRA X ON QUBIT 1 (2nd qubit)
    for i in range(n_qubits):
        qc.h(i)
    qc.x(0); qc.x(1)
    qc.cz(0, 1)
    qc.x(1); qc.x(0)  # Standard unflip
    qc.x(1)  # ← BUG: Extra X specifically on 2nd qubit (qubit 1)
    for i in range(n_qubits):
        qc.h(i)
    
    qc.measure(range(n_qubits), range(n_qubits))
    return qc

# Create circuits
grover_correct = create_grover_circuit(2)
grover_buggy = create_buggy_grover_circuit(2)

print("Grover Correct:")
print(grover_correct.draw(output='text'))
print("\nGrover Buggy:")
print(grover_buggy.draw(output='text'))

Grover Correct:
     ┌───┐┌───┐   ┌───┐┌───┐┌───┐   ┌───┐┌───┐┌─┐   
q_0: ┤ H ├┤ X ├─■─┤ X ├┤ H ├┤ X ├─■─┤ X ├┤ H ├┤M├───
     ├───┤├───┤ │ ├───┤├───┤├───┤ │ ├───┤├───┤└╥┘┌─┐
q_1: ┤ H ├┤ X ├─■─┤ X ├┤ H ├┤ X ├─■─┤ X ├┤ H ├─╫─┤M├
     └───┘└───┘   └───┘└───┘└───┘   └───┘└───┘ ║ └╥┘
c: 2/══════════════════════════════════════════╩══╩═
                                               0  1 

Grover Buggy:
     ┌───┐┌───┐   ┌───┐┌───┐┌───┐   ┌───┐┌───┐     ┌─┐   
q_0: ┤ H ├┤ X ├─■─┤ X ├┤ H ├┤ X ├─■─┤ X ├┤ H ├─────┤M├───
     ├───┤├───┤ │ ├───┤├───┤├───┤ │ ├───┤├───┤┌───┐└╥┘┌─┐
q_1: ┤ H ├┤ X ├─■─┤ X ├┤ H ├┤ X ├─■─┤ X ├┤ X ├┤ H ├─╫─┤M├
     └───┘└───┘   └───┘└───┘└───┘   └───┘└───┘└───┘ ║ └╥┘
c: 2/═══════════════════════════════════════════════╩══╩═
                                                    0  1 


In [7]:
shots = 8192
n_qubits_ghz = 3

print("\n" + "="*80)
print("GHZ STATE SIMULATION")
print("="*80)

# Scenario 1: GHZ on ideal simulator
job_ghz_ideal = simulator_ideal.run(transpile(ghz_correct, simulator_ideal), shots=shots)
counts_ghz_ideal = job_ghz_ideal.result().get_counts()
probs_ghz_ideal = get_probability_distribution(counts_ghz_ideal, n_qubits_ghz)

# Scenario 2: GHZ on noisy hardware
job_ghz_noisy = simulator_noisy.run(transpile(ghz_correct, simulator_noisy), shots=shots)
counts_ghz_noisy = job_ghz_noisy.result().get_counts()
probs_ghz_noisy = get_probability_distribution(counts_ghz_noisy, n_qubits_ghz)


# Scenario 3: Buggy GHZ on noisy hardware
job_ghz_buggy_noisy = simulator_noisy.run(
    transpile(ghz_buggy, simulator_noisy),
    shots=shots
)
counts_ghz_buggy_noisy = job_ghz_buggy_noisy.result().get_counts()
probs_ghz_buggy_noisy = get_probability_distribution(
    counts_ghz_buggy_noisy,
    n_qubits_ghz
)


# Scenario 4: Buggy GHZ on noisy hardware
job_ghz_buggy_noisy = simulator_noisy.run(transpile(ghz_buggy, simulator_noisy), shots=shots)
counts_ghz_buggy_noisy = job_ghz_buggy_noisy.result().get_counts()
probs_ghz_buggy_noisy = get_probability_distribution(counts_ghz_buggy_noisy, n_qubits_ghz)

# Calculate KL divergences
kl_ghz_noise = kl_divergence(probs_ghz_noisy, probs_ghz_ideal)
kl_ghz_bug = kl_divergence(probs_ghz_buggy_noisy, probs_ghz_ideal)
detection_ratio_ghz = kl_ghz_bug / kl_ghz_noise
threshold_ghz_sim = kl_ghz_noise * 3

print(f"\nGHZ State Simulation Results:")
print(f" Hardware Noise KL: {kl_ghz_noise:.4f}")
print(f" Software Bug KL: {kl_ghz_bug:.4f}")
print(f" Detection Ratio: {detection_ratio_ghz:.2f}x")
print(f" Adaptive Threshold: {threshold_ghz_sim:.2f}")

if kl_ghz_bug > threshold_ghz_sim:
    print(f" Verdict: ✓ BUG DETECTED (KL {kl_ghz_bug:.4f} > {threshold_ghz_sim:.2f})")
else:
    print(f" Verdict: ✗ No bug (KL {kl_ghz_bug:.4f} ≤ {threshold_ghz_sim:.2f})")


GHZ STATE SIMULATION

GHZ State Simulation Results:
 Hardware Noise KL: 0.8747
 Software Bug KL: 21.6916
 Detection Ratio: 24.80x
 Adaptive Threshold: 2.62
 Verdict: ✓ BUG DETECTED (KL 21.6916 > 2.62)


In [8]:
n_qubits_w = 3

print("\n" + "="*80)
print("W STATE SIMULATION")
print("="*80)

# Scenario 1: W state on ideal simulator
job_w_ideal = simulator_ideal.run(transpile(w_correct, simulator_ideal), shots=shots)
counts_w_ideal = job_w_ideal.result().get_counts()
probs_w_ideal = get_probability_distribution(counts_w_ideal, n_qubits_w)

# Scenario 2: W state on noisy hardware
job_w_noisy = simulator_noisy.run(transpile(w_correct, simulator_noisy), shots=shots)
counts_w_noisy = job_w_noisy.result().get_counts()
probs_w_noisy = get_probability_distribution(counts_w_noisy, n_qubits_w)


# Scenario 3: Buggy W state on noisy hardware
job_w_buggy_noisy = simulator_noisy.run(
    transpile(w_buggy, simulator_noisy),
    shots=shots
)
counts_w_buggy_noisy = job_w_buggy_noisy.result().get_counts()
probs_w_buggy_noisy = get_probability_distribution(
    counts_w_buggy_noisy,
    n_qubits_w
)


# Scenario 4: Buggy W state on noisy hardware
job_w_buggy_noisy = simulator_noisy.run(transpile(w_buggy, simulator_noisy), shots=shots)
counts_w_buggy_noisy = job_w_buggy_noisy.result().get_counts()
probs_w_buggy_noisy = get_probability_distribution(counts_w_buggy_noisy, n_qubits_w)

# Calculate KL divergences
kl_w_noise = kl_divergence(probs_w_noisy, probs_w_ideal)
kl_w_bug = kl_divergence(probs_w_buggy_noisy, probs_w_ideal)
detection_ratio_w = kl_w_bug / kl_w_noise
threshold_w_sim = kl_w_noise * 3

print(f"\nW State Simulation Results:")
print(f" Hardware Noise KL: {kl_w_noise:.4f}")
print(f" Software Bug KL: {kl_w_bug:.4f}")
print(f" Detection Ratio: {detection_ratio_w:.2f}x")
print(f" Adaptive Threshold: {threshold_w_sim:.2f}")

if kl_w_bug > threshold_w_sim:
    print(f" Verdict: ✓ BUG DETECTED (KL {kl_w_bug:.4f} > {threshold_w_sim:.2f})")
else:
    print(f" Verdict: ✗ No bug (KL {kl_w_bug:.4f} ≤ {threshold_w_sim:.2f})")


W STATE SIMULATION

W State Simulation Results:
 Hardware Noise KL: 0.9574
 Software Bug KL: 21.0956
 Detection Ratio: 22.03x
 Adaptive Threshold: 2.87
 Verdict: ✓ BUG DETECTED (KL 21.0956 > 2.87)


In [9]:
n_qubits_bell = 2

print("\n" + "="*80)
print("BELL STATE SIMULATION")
print("="*80)

# Scenario 1: Bell state on ideal simulator
job_bell_ideal = simulator_ideal.run(transpile(bell_correct, simulator_ideal), shots=shots)
counts_bell_ideal = job_bell_ideal.result().get_counts()
probs_bell_ideal = get_probability_distribution(counts_bell_ideal, n_qubits_bell)

# Scenario 2: Bell state on noisy hardware
job_bell_noisy = simulator_noisy.run(transpile(bell_correct, simulator_noisy), shots=shots)
counts_bell_noisy = job_bell_noisy.result().get_counts()
probs_bell_noisy = get_probability_distribution(counts_bell_noisy, n_qubits_bell)


# Scenario 3: Buggy Bell state on noisy hardware
job_bell_buggy_noisy = simulator_noisy.run(
    transpile(bell_buggy, simulator_noisy),
    shots=shots
)
counts_bell_buggy_noisy = job_bell_buggy_noisy.result().get_counts()
probs_bell_buggy_noisy = get_probability_distribution(
    counts_bell_buggy_noisy,
    n_qubits_bell
)


# Scenario 4: Buggy Bell state on noisy hardware
job_bell_buggy_noisy = simulator_noisy.run(transpile(bell_buggy, simulator_noisy), shots=shots)
counts_bell_buggy_noisy = job_bell_buggy_noisy.result().get_counts()
probs_bell_buggy_noisy = get_probability_distribution(counts_bell_buggy_noisy, n_qubits_bell)

# Calculate KL divergences
kl_bell_noise = kl_divergence(probs_bell_noisy, probs_bell_ideal)
kl_bell_bug = kl_divergence(probs_bell_buggy_noisy, probs_bell_ideal)
detection_ratio_bell = kl_bell_bug / kl_bell_noise
threshold_bell_sim = kl_bell_noise * 3

print(f"\nBell State Simulation Results:")
print(f" Hardware Noise KL: {kl_bell_noise:.4f}")
print(f" Software Bug KL: {kl_bell_bug:.4f}")
print(f" Detection Ratio: {detection_ratio_bell:.2f}x")
print(f" Adaptive Threshold: {threshold_bell_sim:.2f}")

if kl_bell_bug > threshold_bell_sim:
    print(f" Verdict: ✓ BUG DETECTED (KL {kl_bell_bug:.4f} > {threshold_bell_sim:.2f})")
else:
    print(f" Verdict: ✗ No bug (KL {kl_bell_bug:.4f} ≤ {threshold_bell_sim:.2f})")


BELL STATE SIMULATION

Bell State Simulation Results:
 Hardware Noise KL: 0.3745
 Software Bug KL: 21.5085
 Detection Ratio: 57.43x
 Adaptive Threshold: 1.12
 Verdict: ✓ BUG DETECTED (KL 21.5085 > 1.12)


In [10]:
# Grover simulation (add here)
n_qubits_grover = 2
print("\n" + "="*80)
print("GROVER SEARCH SIMULATION")
print("="*80)

job_grover_ideal = simulator_ideal.run(transpile(grover_correct, simulator_ideal), shots=shots)
counts_grover_ideal = job_grover_ideal.result().get_counts()
probs_grover_ideal = get_probability_distribution(counts_grover_ideal, n_qubits_grover)

job_grover_noisy = simulator_noisy.run(transpile(grover_correct, simulator_noisy), shots=shots)
counts_grover_noisy = job_grover_noisy.result().get_counts()
probs_grover_noisy = get_probability_distribution(counts_grover_noisy, n_qubits_grover)

job_grover_buggy_noisy = simulator_noisy.run(transpile(grover_buggy, simulator_noisy), shots=shots)
counts_grover_buggy_noisy = job_grover_buggy_noisy.result().get_counts()
probs_grover_buggy_noisy = get_probability_distribution(counts_grover_buggy_noisy, n_qubits_grover)

kl_grover_noise = kl_divergence(probs_grover_noisy, probs_grover_ideal)
kl_grover_bug = kl_divergence(probs_grover_buggy_noisy, probs_grover_ideal)
detection_ratio_grover = kl_grover_bug / kl_grover_noise
threshold_grover_sim = kl_grover_noise * 3

print(f"\nGrover Results:")
print(f" Noise KL: {kl_grover_noise:.4f}")
print(f" Bug KL: {kl_grover_bug:.4f}")
print(f" Ratio: {detection_ratio_grover:.2f}x")
print(f" Threshold: {threshold_grover_sim:.2f}")
print(f" Verdict: {'✓ BUG DETECTED' if kl_grover_bug > threshold_grover_sim else '✗ No bug'}")


GROVER SEARCH SIMULATION

Grover Results:
 Noise KL: 2.3441
 Bug KL: 2.1619
 Ratio: 0.92x
 Threshold: 7.03
 Verdict: ✗ No bug


In [11]:
print("\n" + "="*80)
print("SIMULATION COMPARISON: GHZ vs W vs BELL")
print("="*80)

print(f"\n{'Circuit':<15} {'Qubits':<10} {'Noise KL':<12} {'Bug KL':<12} {'Ratio':<10} {'Detection'}")
print("-"*80)
print(f"{'GHZ State':<15} {3:<10} {kl_ghz_noise:<12.4f} {kl_ghz_bug:<12.4f} {detection_ratio_ghz:>8.2f}x  {'✓' if detection_ratio_ghz > 2 else '✗'}")
print(f"{'W State':<15} {3:<10} {kl_w_noise:<12.4f} {kl_w_bug:<12.4f} {detection_ratio_w:>8.2f}x  {'✓' if detection_ratio_w > 2 else '✗'}")
print(f"{'Bell State':<15} {2:<10} {kl_bell_noise:<12.4f} {kl_bell_bug:<12.4f} {detection_ratio_bell:>8.2f}x  {'✓' if detection_ratio_bell > 2 else '✗'}")
print(f"{'Grover':<15} {2:<10} {kl_grover_noise:<12.4f} {kl_grover_bug:<12.4f} {detection_ratio_grover:>8.2f}x {'✓' if detection_ratio_grover > 2 else '✗'}")


print("\n" + "="*80)
print("KEY INSIGHTS FROM SIMULATION")
print("="*80)
print("\n1. Detection Ratio > 2.0x indicates bugs produce significantly")
print("   larger KL-divergence than hardware noise")
print("\n2. W state may show different noise sensitivity due to")
print("   robustness against particle loss")
print("\n3. Bell state (2 qubits) has less accumulated noise,")
print("   potentially cleaner bug detection")


SIMULATION COMPARISON: GHZ vs W vs BELL

Circuit         Qubits     Noise KL     Bug KL       Ratio      Detection
--------------------------------------------------------------------------------
GHZ State       3          0.8747       21.6916         24.80x  ✓
W State         3          0.9574       21.0956         22.03x  ✓
Bell State      2          0.3745       21.5085         57.43x  ✓
Grover          2          2.3441       2.1619           0.92x ✗

KEY INSIGHTS FROM SIMULATION

1. Detection Ratio > 2.0x indicates bugs produce significantly
   larger KL-divergence than hardware noise

2. W state may show different noise sensitivity due to
   robustness against particle loss

3. Bell state (2 qubits) has less accumulated noise,
   potentially cleaner bug detection


In [12]:
print("\n" + "="*80)
print("CONNECTING TO IBM QUANTUM HARDWARE")
print("="*80)

from kaggle_secrets import UserSecretsClient
from qiskit_ibm_runtime import QiskitRuntimeService

user_secrets = UserSecretsClient()
token = user_secrets.get_secret("IBM_QUANTUM_TOKEN")

service = QiskitRuntimeService(
    token=token,
    instance="open-instance"  # explicit free-plan instance
)

# Select least busy backend
backend = service.least_busy(
    simulator=False,
    operational=True,
    min_num_qubits=3
)

print(f"\nSelected backend: {backend.name}")
print(f"Number of qubits: {backend.num_qubits}")
print(f"Pending jobs: {backend.status().pending_jobs}")

# Create pass manager for transpilation
pm = generate_preset_pass_manager(
    backend=backend,
    optimization_level=1,
    translation_method='translator'
)

print("Transpilation pass manager ready")


CONNECTING TO IBM QUANTUM HARDWARE


qiskit_runtime_service._discover_account:WARNING:2026-01-09 21:33:26,115: Loading account with the given token. A saved account will not be used.



Selected backend: ibm_fez
Number of qubits: 156
Pending jobs: 0
Transpilation pass manager ready


In [13]:
print("\n" + "="*80)
print("GHZ STATE ON REAL QUANTUM HARDWARE")
print("="*80)

# Transpile GHZ circuits
isa_ghz_correct = pm.run(ghz_correct)
isa_ghz_buggy = pm.run(ghz_buggy)

# Run both circuits on real hardware
sampler_ghz = Sampler(mode=backend)
job_ghz_hw = sampler_ghz.run(
    pubs=[isa_ghz_correct, isa_ghz_buggy],
    shots=shots
)
result_ghz_hw = job_ghz_hw.result()

# Extract counts - access measurement data correctly
# Try different methods to get counts
try:
    # Method 1: Direct access to classical register
    counts_ghz_hw_correct = result_ghz_hw[0].data.c.get_counts()
    counts_ghz_hw_buggy = result_ghz_hw[1].data.c.get_counts()
except AttributeError:
    try:
        # Method 2: Access through the first classical register name
        reg_name = list(result_ghz_hw[0].data.keys())[0]
        counts_ghz_hw_correct = result_ghz_hw[0].data[reg_name].get_counts()
        counts_ghz_hw_buggy = result_ghz_hw[1].data[reg_name].get_counts()
    except:
        # Method 3: If get_counts() doesn't exist, construct manually
        counts_ghz_hw_correct = {}
        counts_ghz_hw_buggy = {}
        
        # Get bitstring data
        for bitstring, count in zip(result_ghz_hw[0].data.get_bitstrings(), 
                                     result_ghz_hw[0].data.get_counts()):
            counts_ghz_hw_correct[bitstring] = count
        
        for bitstring, count in zip(result_ghz_hw[1].data.get_bitstrings(), 
                                     result_ghz_hw[1].data.get_counts()):
            counts_ghz_hw_buggy[bitstring] = count

# Extract probability distributions
probs_ghz_hw_correct = get_probability_distribution(
    counts_ghz_hw_correct,
    n_qubits_ghz
)
probs_ghz_hw_buggy = get_probability_distribution(
    counts_ghz_hw_buggy,
    n_qubits_ghz
)

# Calculate KL divergences
kl_ghz_hw_noise = kl_divergence(probs_ghz_hw_correct, probs_ghz_ideal)
kl_ghz_hw_bug = kl_divergence(probs_ghz_hw_buggy, probs_ghz_ideal)
detection_ratio_ghz_hw = kl_ghz_hw_bug / kl_ghz_hw_noise

print(f"\nBackend: {backend.name}")
print(f"Shots: {shots}")
print(f"\nGHZ State Real Hardware Results:")
print(f" Real hardware noise KL: {kl_ghz_hw_noise:.4f}")
print(f" Real hardware bug KL: {kl_ghz_hw_bug:.4f}")
print(f" Real detection ratio: {detection_ratio_ghz_hw:.2f}x")

if detection_ratio_ghz_hw > 1.5:
    print(f" Verdict: ✓ BUG DETECTED (ratio {detection_ratio_ghz_hw:.2f}x > 1.5)")
else:
    print(f" Verdict: ✗ Inconclusive (ratio {detection_ratio_ghz_hw:.2f}x ≤ 1.5)")


GHZ STATE ON REAL QUANTUM HARDWARE

Backend: ibm_fez
Shots: 8192

GHZ State Real Hardware Results:
 Real hardware noise KL: 1.2143
 Real hardware bug KL: 21.6234
 Real detection ratio: 17.81x
 Verdict: ✓ BUG DETECTED (ratio 17.81x > 1.5)


In [14]:
print("\n" + "="*80)
print("W STATE ON REAL QUANTUM HARDWARE")
print("="*80)

# Transpile W state circuits
isa_w_correct = pm.run(w_correct)
isa_w_buggy = pm.run(w_buggy)

# Run both circuits on real hardware
sampler_w = Sampler(mode=backend)
job_w_hw = sampler_w.run(
    pubs=[isa_w_correct, isa_w_buggy],
    shots=shots
)
result_w_hw = job_w_hw.result()

# Extract counts
try:
    counts_w_hw_correct = result_w_hw[0].data.c.get_counts()
    counts_w_hw_buggy = result_w_hw[1].data.c.get_counts()
except AttributeError:
    reg_name = list(result_w_hw[0].data.keys())[0]
    counts_w_hw_correct = result_w_hw[0].data[reg_name].get_counts()
    counts_w_hw_buggy = result_w_hw[1].data[reg_name].get_counts()

# Extract probability distributions
probs_w_hw_correct = get_probability_distribution(
    counts_w_hw_correct,
    n_qubits_w
)
probs_w_hw_buggy = get_probability_distribution(
    counts_w_hw_buggy,
    n_qubits_w
)

# Calculate KL divergences
kl_w_hw_noise = kl_divergence(probs_w_hw_correct, probs_w_ideal)
kl_w_hw_bug = kl_divergence(probs_w_hw_buggy, probs_w_ideal)
detection_ratio_w_hw = kl_w_hw_bug / kl_w_hw_noise

print(f"\nBackend: {backend.name}")
print(f"Shots: {shots}")
print(f"\nW State Real Hardware Results:")
print(f" Real hardware noise KL: {kl_w_hw_noise:.4f}")
print(f" Real hardware bug KL: {kl_w_hw_bug:.4f}")
print(f" Real detection ratio: {detection_ratio_w_hw:.2f}x")

if detection_ratio_w_hw > 1.5:
    print(f" Verdict: ✓ BUG DETECTED (ratio {detection_ratio_w_hw:.2f}x > 1.5)")
else:
    print(f" Verdict: ✗ Inconclusive (ratio {detection_ratio_w_hw:.2f}x ≤ 1.5)")


W STATE ON REAL QUANTUM HARDWARE

Backend: ibm_fez
Shots: 8192

W State Real Hardware Results:
 Real hardware noise KL: 0.7758
 Real hardware bug KL: 21.1946
 Real detection ratio: 27.32x
 Verdict: ✓ BUG DETECTED (ratio 27.32x > 1.5)


In [15]:
print("\n" + "="*80)
print("BELL STATE ON REAL QUANTUM HARDWARE")
print("="*80)

# Transpile Bell state circuits
isa_bell_correct = pm.run(bell_correct)
isa_bell_buggy = pm.run(bell_buggy)

# Run both circuits on real hardware
sampler_bell = Sampler(mode=backend)
job_bell_hw = sampler_bell.run(
    pubs=[isa_bell_correct, isa_bell_buggy],
    shots=shots
)
result_bell_hw = job_bell_hw.result()

# Extract counts
try:
    counts_bell_hw_correct = result_bell_hw[0].data.c.get_counts()
    counts_bell_hw_buggy = result_bell_hw[1].data.c.get_counts()
except AttributeError:
    reg_name = list(result_bell_hw[0].data.keys())[0]
    counts_bell_hw_correct = result_bell_hw[0].data[reg_name].get_counts()
    counts_bell_hw_buggy = result_bell_hw[1].data[reg_name].get_counts()

# Extract probability distributions
probs_bell_hw_correct = get_probability_distribution(
    counts_bell_hw_correct,
    n_qubits_bell
)
probs_bell_hw_buggy = get_probability_distribution(
    counts_bell_hw_buggy,
    n_qubits_bell
)

# Calculate KL divergences
kl_bell_hw_noise = kl_divergence(probs_bell_hw_correct, probs_bell_ideal)
kl_bell_hw_bug = kl_divergence(probs_bell_hw_buggy, probs_bell_ideal)
detection_ratio_bell_hw = kl_bell_hw_bug / kl_bell_hw_noise

print(f"\nBackend: {backend.name}")
print(f"Shots: {shots}")
print(f"\nBell State Real Hardware Results:")
print(f" Real hardware noise KL: {kl_bell_hw_noise:.4f}")
print(f" Real hardware bug KL: {kl_bell_hw_bug:.4f}")
print(f" Real detection ratio: {detection_ratio_bell_hw:.2f}x")

if detection_ratio_bell_hw > 1.5:
    print(f" Verdict: ✓ BUG DETECTED (ratio {detection_ratio_bell_hw:.2f}x > 1.5)")
else:
    print(f" Verdict: ✗ Inconclusive (ratio {detection_ratio_bell_hw:.2f}x ≤ 1.5)")


BELL STATE ON REAL QUANTUM HARDWARE

Backend: ibm_fez
Shots: 8192

Bell State Real Hardware Results:
 Real hardware noise KL: 0.9583
 Real hardware bug KL: 21.0485
 Real detection ratio: 21.96x
 Verdict: ✓ BUG DETECTED (ratio 21.96x > 1.5)


In [16]:
print("\n" + "="*80)
print("GROVER ON REAL HARDWARE")
print("="*80)

isa_grover_correct = pm.run(grover_correct)
isa_grover_buggy = pm.run(grover_buggy)

sampler_grover = Sampler(mode=backend)
job_grover_hw = sampler_grover.run(pubs=[isa_grover_correct, isa_grover_buggy], shots=shots)
result_grover_hw = job_grover_hw.result()

# Extract counts (same pattern as others)
try:
    counts_grover_hw_correct = result_grover_hw[0].data.c.get_counts()
    counts_grover_hw_buggy = result_grover_hw[1].data.c.get_counts()
except:
    reg_name = list(result_grover_hw[0].data.keys())[0]
    counts_grover_hw_correct = result_grover_hw[0].data[reg_name].get_counts()
    counts_grover_hw_buggy = result_grover_hw[1].data[reg_name].get_counts()

probs_grover_hw_correct = get_probability_distribution(counts_grover_hw_correct, n_qubits_grover)
probs_grover_hw_buggy = get_probability_distribution(counts_grover_hw_buggy, n_qubits_grover)

kl_grover_hw_noise = kl_divergence(probs_grover_hw_correct, probs_grover_ideal)
kl_grover_hw_bug = kl_divergence(probs_grover_hw_buggy, probs_grover_ideal)
detection_ratio_grover_hw = kl_grover_hw_bug / kl_grover_hw_noise

print(f"\nGrover Hardware Results:")
print(f" Noise KL: {kl_grover_hw_noise:.4f}")
print(f" Bug KL: {kl_grover_hw_bug:.4f}")
print(f" Ratio: {detection_ratio_grover_hw:.2f}x")
print(f" Verdict: {'✓ BUG DETECTED' if detection_ratio_grover_hw > 1.5 else '✗ Inconclusive'}")


GROVER ON REAL HARDWARE

Grover Hardware Results:
 Noise KL: 0.9016
 Bug KL: 0.9885
 Ratio: 1.10x
 Verdict: ✗ Inconclusive


In [17]:
print("\n" + "="*80)
print("FINAL COMPARISON: SIMULATION vs REAL HARDWARE")
print("="*80)

print("\n" + "-"*80)
print("SIMULATED HARDWARE (Depolarizing Noise Model)")
print("-"*80)
print(f"{'Circuit':<15} {'Qubits':<10} {'Noise KL':<12} {'Bug KL':<12} {'Ratio':<10} {'Detection'}")
print("-"*80)
print(f"{'GHZ':<15} {3:<10} {kl_ghz_noise:<12.4f} {kl_ghz_bug:<12.4f} {detection_ratio_ghz:>8.2f}x  {'✓' if detection_ratio_ghz > 2 else '✗'}")
print(f"{'W State':<15} {3:<10} {kl_w_noise:<12.4f} {kl_w_bug:<12.4f} {detection_ratio_w:>8.2f}x  {'✓' if detection_ratio_w > 2 else '✗'}")
print(f"{'Bell':<15} {2:<10} {kl_bell_noise:<12.4f} {kl_bell_bug:<12.4f} {detection_ratio_bell:>8.2f}x  {'✓' if detection_ratio_bell > 2 else '✗'}")
print(f"{'Grover':<15} {2:<10} {kl_grover_noise:<12.4f} {kl_grover_bug:<12.4f} {detection_ratio_grover:>8.2f}x {'✓' if detection_ratio_grover > 2 else '✗'}")

print("\n" + "-"*80)
print(f"REAL HARDWARE ({backend.name})")
print("-"*80)
print(f"{'Circuit':<15} {'Qubits':<10} {'Noise KL':<12} {'Bug KL':<12} {'Ratio':<10} {'Detection'}")
print("-"*80)
print(f"{'GHZ':<15} {3:<10} {kl_ghz_hw_noise:<12.4f} {kl_ghz_hw_bug:<12.4f} {detection_ratio_ghz_hw:>8.2f}x  {'✓' if detection_ratio_ghz_hw > 1.5 else '✗'}")
print(f"{'W State':<15} {3:<10} {kl_w_hw_noise:<12.4f} {kl_w_hw_bug:<12.4f} {detection_ratio_w_hw:>8.2f}x  {'✓' if detection_ratio_w_hw > 1.5 else '✗'}")
print(f"{'Bell':<15} {2:<10} {kl_bell_hw_noise:<12.4f} {kl_bell_hw_bug:<12.4f} {detection_ratio_bell_hw:>8.2f}x  {'✓' if detection_ratio_bell_hw > 1.5 else '✗'}")
print(f"{'Grover':<15} {2:<10} {kl_grover_hw_noise:<12.4f} {kl_grover_hw_bug:<12.4f} {detection_ratio_grover_hw:>8.2f}x {'✓' if detection_ratio_grover_hw > 1.5 else '✗'}")


FINAL COMPARISON: SIMULATION vs REAL HARDWARE

--------------------------------------------------------------------------------
SIMULATED HARDWARE (Depolarizing Noise Model)
--------------------------------------------------------------------------------
Circuit         Qubits     Noise KL     Bug KL       Ratio      Detection
--------------------------------------------------------------------------------
GHZ             3          0.8747       21.6916         24.80x  ✓
W State         3          0.9574       21.0956         22.03x  ✓
Bell            2          0.3745       21.5085         57.43x  ✓
Grover          2          2.3441       2.1619           0.92x ✗

--------------------------------------------------------------------------------
REAL HARDWARE (ibm_fez)
--------------------------------------------------------------------------------
Circuit         Qubits     Noise KL     Bug KL       Ratio      Detection
-----------------------------------------------------------------

In [18]:
print("\n" + "="*80)
print("DETAILED PROBABILITY DISTRIBUTIONS")
print("="*80)

# GHZ (3 qubits) - existing code unchanged
print("\nGHZ STATE (3 qubits):")
states_3q = [f"|{i:03b}⟩" for i in range(8)]
print(f"{'State':<10} {'Ideal':<12} {'Sim Noisy':<12} {'Sim Buggy':<12} {'HW Correct':<12} {'HW Buggy':<12}")
print("-" * 90)
for i in range(8):
    print(f"{states_3q[i]:<10} {probs_ghz_ideal[i]:>10.4f} {probs_ghz_noisy[i]:>10.4f} {probs_ghz_buggy_noisy[i]:>10.4f} {probs_ghz_hw_correct[i]:>10.4f} {probs_ghz_hw_buggy[i]:>10.4f}")

# W (3 qubits) - existing unchanged
print("\n\nW STATE (3 qubits):")
print(f"{'State':<10} {'Ideal':<12} {'Sim Noisy':<12} {'Sim Buggy':<12} {'HW Correct':<12} {'HW Buggy':<12}")
print("-" * 90)
for i in range(8):
    print(f"{states_3q[i]:<10} {probs_w_ideal[i]:>10.4f} {probs_w_noisy[i]:>10.4f} {probs_w_buggy_noisy[i]:>10.4f} {probs_w_hw_correct[i]:>10.4f} {probs_w_hw_buggy[i]:>10.4f}")

# Bell (2 qubits) - existing unchanged
print("\n\nBELL STATE (2 qubits):")
states_2q = [f"|{i:02b}⟩" for i in range(4)]
print(f"{'State':<10} {'Ideal':<12} {'Sim Noisy':<12} {'Sim Buggy':<12} {'HW Correct':<12} {'HW Buggy':<12}")
print("-" * 90)
for i in range(4):
    print(f"{states_2q[i]:<10} {probs_bell_ideal[i]:>10.4f} {probs_bell_noisy[i]:>10.4f} {probs_bell_buggy_noisy[i]:>10.4f} {probs_bell_hw_correct[i]:>10.4f} {probs_bell_hw_buggy[i]:>10.4f}")

# GROVER (2 qubits) - NEW SECTION
print("\n\nGROVER SEARCH (2 qubits):")
print(f"{'State':<10} {'Ideal':<12} {'Sim Noisy':<12} {'Sim Buggy':<12} {'HW Correct':<12} {'HW Buggy':<12}")
print("-" * 90)
for i in range(4):
    print(f"{states_2q[i]:<10} {probs_grover_ideal[i]:>10.4f} {probs_grover_noisy[i]:>10.4f} {probs_grover_buggy_noisy[i]:>10.4f} {probs_grover_hw_correct[i]:>10.4f} {probs_grover_hw_buggy[i]:>10.4f}")


DETAILED PROBABILITY DISTRIBUTIONS

GHZ STATE (3 qubits):
State      Ideal        Sim Noisy    Sim Buggy    HW Correct   HW Buggy    
------------------------------------------------------------------------------------------
|000⟩          0.4971     0.4742     0.0083     0.4893     0.0121
|001⟩          0.0000     0.0134     0.0056     0.0129     0.0052
|010⟩          0.0000     0.0052     0.4689     0.0037     0.4750
|011⟩          0.0000     0.0056     0.0126     0.0044     0.0148
|100⟩          0.0000     0.0045     0.0140     0.0043     0.0288
|101⟩          0.0000     0.0062     0.4773     0.0104     0.4536
|110⟩          0.0000     0.0153     0.0048     0.0321     0.0051
|111⟩          0.5029     0.4755     0.0085     0.4430     0.0054


W STATE (3 qubits):
State      Ideal        Sim Noisy    Sim Buggy    HW Correct   HW Buggy    
------------------------------------------------------------------------------------------
|000⟩          0.3373     0.3152     0.0103     0.3232   

In [19]:
print("\n" + "="*80)
print("RESEARCH CONCLUSIONS")
print("="*80)

print("\n1. HYPOTHESIS VALIDATION:")
print("   Can KL-divergence distinguish software bugs from hardware noise?")

sim_success = sum([
    detection_ratio_ghz > 2,
    detection_ratio_w > 2,
    detection_ratio_bell > 2,
    detection_ratio_grover > 2
])

hw_success = sum([
    detection_ratio_ghz_hw > 1.5,
    detection_ratio_w_hw > 1.5,
    detection_ratio_bell_hw > 1.5,
    detection_ratio_grover_hw > 1.5
])

print(f"\n Simulation: {sim_success}/4 circuits successfully detected bugs")
print(f" Real Hardware: {hw_success}/4 circuits successfully detected bugs")

print("\n2. ENTANGLEMENT STRUCTURE IMPACT:")
print(f"   - GHZ (maximally entangled): Sim ratio {detection_ratio_ghz:.2f}x, HW ratio {detection_ratio_ghz_hw:.2f}x")
print(f"   - W (robust entangled): Sim ratio {detection_ratio_w:.2f}x, HW ratio {detection_ratio_w_hw:.2f}x")
print(f"   - Bell (simple entangled): Sim ratio {detection_ratio_bell:.2f}x, HW ratio {detection_ratio_bell_hw:.2f}x")
print(f"   - Grover (algorithmic): Sim ratio {detection_ratio_grover:.2f}x, HW ratio {detection_ratio_grover_hw:.2f}x")

print("\n3. NOISE SENSITIVITY:")
print(f"   - GHZ noise: Sim {kl_ghz_noise:.4f}, HW {kl_ghz_hw_noise:.4f}")
print(f"   - W noise: Sim {kl_w_noise:.4f}, HW {kl_w_hw_noise:.4f}")
print(f"   - Bell noise: Sim {kl_bell_noise:.4f}, HW {kl_bell_hw_noise:.4f}")
print(f"   - Grover noise: Sim {kl_grover_noise:.4f}, HW {kl_grover_hw_noise:.4f}")

print("\n4. W STATE ROBUSTNESS OBSERVATION:")
if kl_w_hw_noise < kl_ghz_hw_noise:
    print("   W state shows LOWER noise KL than GHZ (confirms robustness)")
else:
    print("   W state noise similar to GHZ (hardware-specific)")

print("\n5. PRACTICAL RECOMMENDATIONS:")
print("   - Use adaptive thresholds (3x noise KL for simulation)")
print("   - Use detection ratios > 1.5x for real hardware")
print("   - Test multiple circuit types for robustness")
print("   - W states reliable for noisy devices")
print("   - Grover validates algorithmic circuits")
print("   - Bell provides clean 2-qubit baseline")

print("\n" + "="*80)
print("EXPERIMENT COMPLETE")
print("="*80)


RESEARCH CONCLUSIONS

1. HYPOTHESIS VALIDATION:
   Can KL-divergence distinguish software bugs from hardware noise?

 Simulation: 3/4 circuits successfully detected bugs
 Real Hardware: 3/4 circuits successfully detected bugs

2. ENTANGLEMENT STRUCTURE IMPACT:
   - GHZ (maximally entangled): Sim ratio 24.80x, HW ratio 17.81x
   - W (robust entangled): Sim ratio 22.03x, HW ratio 27.32x
   - Bell (simple entangled): Sim ratio 57.43x, HW ratio 21.96x
   - Grover (algorithmic): Sim ratio 0.92x, HW ratio 1.10x

3. NOISE SENSITIVITY:
   - GHZ noise: Sim 0.8747, HW 1.2143
   - W noise: Sim 0.9574, HW 0.7758
   - Bell noise: Sim 0.3745, HW 0.9583
   - Grover noise: Sim 2.3441, HW 0.9016

4. W STATE ROBUSTNESS OBSERVATION:
   W state shows LOWER noise KL than GHZ (confirms robustness)

5. PRACTICAL RECOMMENDATIONS:
   - Use adaptive thresholds (3x noise KL for simulation)
   - Use detection ratios > 1.5x for real hardware
   - Test multiple circuit types for robustness
   - W states reliable f

In [20]:
print("\n" + "="*80)
print(f"BACKEND CONFIGURATION: {backend.name}")
print("="*80)

# Basic properties
print(f"\nBasic Information:")
print(f" Name: {backend.name}")
print(f" Version: {backend.version}")
print(f" Number of qubits: {backend.num_qubits}")
print(f" Status: {'Operational' if backend.status().operational else 'Down'}")
print(f" Pending jobs: {backend.status().pending_jobs}")

# Get configuration
config = backend.configuration()
print(f"\nConfiguration:")
print(f" Backend type: {config.backend_name}")
print(f" Simulator: {config.simulator}")
print(f" Local: {config.local}")
print(f" Max shots: {config.max_shots}")
print(f" Max experiments: {config.max_experiments}")
print(f" Basis gates: {config.basis_gates}")

# Coupling map (only show relevant qubits)
print(f"\n Coupling map (first 20 connections):")
coupling_pairs = config.coupling_map[:20]
print(f" {coupling_pairs}")

# Get qubit properties (error rates, T1, T2)
try:
    target = backend.target
    print(f"\nQubit Properties (for qubits used in experiments):")
    
    # For GHZ and W states (3 qubits)
    for qubit in range(min(3, backend.num_qubits)):
        qubit_props = target.qubit_properties[qubit]
        print(f"\n Qubit {qubit}:")
        print(f"  T1 (relaxation): {qubit_props.t1*1e6:.2f} μs")
        print(f"  T2 (dephasing): {qubit_props.t2*1e6:.2f} μs")
        print(f"  Frequency: {qubit_props.frequency/1e9:.4f} GHz")
except Exception as e:
    print(f"\n Qubit properties not available: {e}")

# Gate error rates
try:
    print(f"\nSingle-Qubit Gate Error Rates:")
    for qubit in range(min(3, backend.num_qubits)):
        # X gate errors
        if 'x' in target:
            try:
                x_error = target['x'][(qubit,)].error
                print(f" X gate on qubit {qubit}: {x_error*100:.3f}%")
            except:
                pass
        
        # SX gate errors (if available)
        if 'sx' in target:
            try:
                sx_error = target['sx'][(qubit,)].error
                print(f" SX gate on qubit {qubit}: {sx_error*100:.3f}%")
            except:
                pass
    
    print(f"\nTwo-Qubit Gate Error Rates (CNOT/CX):")
    # Check CX gates between consecutive qubits
    for i in range(min(2, backend.num_qubits - 1)):
        for direction in [(i, i+1), (i+1, i)]:
            if direction in config.coupling_map:
                try:
                    if 'cx' in target:
                        cx_error = target['cx'][direction].error
                        print(f" CX({direction[0]},{direction[1]}): {cx_error*100:.3f}%")
                    elif 'ecr' in target:
                        ecr_error = target['ecr'][direction].error
                        print(f" ECR({direction[0]},{direction[1]}): {ecr_error*100:.3f}%")
                except:
                    pass
                    
except Exception as e:
    print(f"\n Gate error rates not available: {e}")

# Readout errors
try:
    print(f"\nReadout (Measurement) Error Rates:")
    for qubit in range(min(3, backend.num_qubits)):
        meas_error = target['measure'][(qubit,)].error
        print(f" Qubit {qubit}: {meas_error*100:.3f}%")
except Exception as e:
    print(f"\n Readout error rates not available: {e}")

# Calculate average error rates for summary
try:
    print(f"\nAverage Error Rates Summary:")
    
    # Average single-qubit gate error
    sq_errors = []
    for qubit in range(min(3, backend.num_qubits)):
        if 'x' in target:
            try:
                sq_errors.append(target['x'][(qubit,)].error)
            except:
                pass
    
    if sq_errors:
        avg_sq_error = np.mean(sq_errors)
        print(f" Average single-qubit gate error: {avg_sq_error*100:.3f}%")
    
    # Average two-qubit gate error
    tq_errors = []
    for i in range(min(2, backend.num_qubits - 1)):
        if (i, i+1) in config.coupling_map:
            try:
                if 'cx' in target:
                    tq_errors.append(target['cx'][(i, i+1)].error)
            except:
                pass
    
    if tq_errors:
        avg_tq_error = np.mean(tq_errors)
        print(f" Average two-qubit gate error: {avg_tq_error*100:.3f}%")
    
    # Average readout error
    ro_errors = []
    for qubit in range(min(3, backend.num_qubits)):
        try:
            ro_errors.append(target['measure'][(qubit,)].error)
        except:
            pass
    
    if ro_errors:
        avg_ro_error = np.mean(ro_errors)
        print(f" Average readout error: {avg_ro_error*100:.3f}%")
        
except Exception as e:
    print(f"\n Could not calculate average errors: {e}")

print("\n" + "="*80)


BACKEND CONFIGURATION: ibm_fez

Basic Information:
 Name: ibm_fez
 Version: 2
 Number of qubits: 156
 Status: Operational
 Pending jobs: 0

Configuration:
 Backend type: ibm_fez
 Simulator: False
 Local: False
 Max shots: 100000
 Max experiments: 300
 Basis gates: ['cz', 'id', 'rz', 'sx', 'x']

 Coupling map (first 20 connections):
 [[0, 1], [1, 0], [1, 2], [2, 1], [2, 3], [3, 2], [3, 4], [3, 16], [4, 3], [4, 5], [5, 4], [5, 6], [6, 5], [6, 7], [7, 6], [7, 8], [7, 17], [8, 7], [8, 9], [9, 8]]

Qubit Properties (for qubits used in experiments):

 Qubit 0:
  T1 (relaxation): 50.95 μs
  T2 (dephasing): 50.08 μs

 Qubit properties not available: unsupported operand type(s) for /: 'NoneType' and 'float'

Single-Qubit Gate Error Rates:
 X gate on qubit 0: 0.162%
 SX gate on qubit 0: 0.162%
 X gate on qubit 1: 0.023%
 SX gate on qubit 1: 0.023%
 X gate on qubit 2: 0.019%
 SX gate on qubit 2: 0.019%

Two-Qubit Gate Error Rates (CNOT/CX):

Readout (Measurement) Error Rates:
 Qubit 0: 1.282%
 Q

In [21]:
print("\n" + "="*80)
print("ERROR BUDGET ANALYSIS")
print("="*80)

print("\nThis analysis estimates the total error in each circuit")
print("based on the number of gates and measured error rates.\n")

try:
    target = backend.target
    
    # Function to estimate circuit error
    def estimate_circuit_error(circuit_name, n_h_gates, n_cx_gates, n_x_gates, n_qubits):
        """
        Estimate total circuit error based on gate counts
        Uses error propagation: total_error ≈ sum of individual gate errors
        """
        total_error = 0
        
        # Single-qubit gate errors (H, X)
        sq_errors = []
        for qubit in range(min(n_qubits, backend.num_qubits)):
            if 'x' in target:
                try:
                    sq_errors.append(target['x'][(qubit,)].error)
                except:
                    pass
        
        if sq_errors:
            avg_sq_error = np.mean(sq_errors)
            total_error += (n_h_gates + n_x_gates) * avg_sq_error
        
        # Two-qubit gate errors (CX)
        tq_errors = []
        for i in range(min(n_qubits - 1, backend.num_qubits - 1)):
            if (i, i+1) in config.coupling_map:
                try:
                    if 'cx' in target:
                        tq_errors.append(target['cx'][(i, i+1)].error)
                except:
                    pass
        
        if tq_errors:
            avg_tq_error = np.mean(tq_errors)
            total_error += n_cx_gates * avg_tq_error
        
        # Readout errors
        ro_errors = []
        for qubit in range(min(n_qubits, backend.num_qubits)):
            try:
                ro_errors.append(target['measure'][(qubit,)].error)
            except:
                pass
        
        if ro_errors:
            total_error += np.sum(ro_errors[:n_qubits])
        
        return total_error
    
    # Estimate errors for each circuit type
    print("Circuit Error Budget Estimates:\n")
    
    # GHZ circuit: 1 H + 2 CX + 3 measurements
    ghz_error = estimate_circuit_error("GHZ", n_h_gates=1, n_cx_gates=2, n_x_gates=0, n_qubits=3)
    print(f"GHZ Circuit (correct):")
    print(f" Gates: 1 H + 2 CX")
    print(f" Estimated total error: {ghz_error*100:.2f}%")
    
    # GHZ buggy: 1 H + 2 CX + 1 X + 3 measurements
    ghz_buggy_error = estimate_circuit_error("GHZ Buggy", n_h_gates=1, n_cx_gates=2, n_x_gates=1, n_qubits=3)
    print(f"\nGHZ Circuit (buggy with X):")
    print(f" Gates: 1 H + 2 CX + 1 X")
    print(f" Estimated total error: {ghz_buggy_error*100:.2f}%")
    
    # W state: 1 RY + 2 X + 1 CRY + 1 CX + 3 measurements
    # Approximate RY and CRY as single-qubit gates
    w_error = estimate_circuit_error("W State", n_h_gates=0, n_cx_gates=2, n_x_gates=3, n_qubits=3)
    print(f"\nW State Circuit (correct):")
    print(f" Gates: 1 RY + 2 X + 1 CRY + 1 CX")
    print(f" Estimated total error: {w_error*100:.2f}%")
    
    # W buggy: same + 1 X
    w_buggy_error = estimate_circuit_error("W State Buggy", n_h_gates=0, n_cx_gates=2, n_x_gates=4, n_qubits=3)
    print(f"\nW State Circuit (buggy with X):")
    print(f" Gates: 1 RY + 3 X + 1 CRY + 1 CX")
    print(f" Estimated total error: {w_buggy_error*100:.2f}%")
    
    # Bell state: 1 H + 1 CX + 2 measurements
    bell_error = estimate_circuit_error("Bell", n_h_gates=1, n_cx_gates=1, n_x_gates=0, n_qubits=2)
    print(f"\nBell State Circuit (correct):")
    print(f" Gates: 1 H + 1 CX")
    print(f" Estimated total error: {bell_error*100:.2f}%")
    
    # Bell buggy: 1 H + 1 CX + 1 X + 2 measurements
    bell_buggy_error = estimate_circuit_error("Bell Buggy", n_h_gates=1, n_cx_gates=1, n_x_gates=1, n_qubits=2)
    print(f"\nBell State Circuit (buggy with X):")
    print(f" Gates: 1 H + 1 CX + 1 X")
    print(f" Estimated total error: {bell_buggy_error*100:.2f}%")
    
    # Grover: 8 H + 6 X + 2 CZ (CZ ≈ 2 CX equivalent) + 2 measurements
    grover_error = estimate_circuit_error("Grover", n_h_gates=8, n_cx_gates=4, n_x_gates=6, n_qubits=2)
    print(f"\nGrover Circuit (correct):")
    print(f" Gates: 8 H + 6 X + 2 CZ")
    print(f" Estimated total error: {grover_error*100:.2f}%")
    
    # Grover buggy: +1 X
    grover_buggy_error = estimate_circuit_error("Grover Buggy", n_h_gates=8, n_cx_gates=4, n_x_gates=7, n_qubits=2)
    print(f"\nGrover Circuit (buggy with X):")
    print(f" Gates: 8 H + 7 X + 2 CZ")
    print(f" Estimated total error: {grover_buggy_error*100:.2f}%")
    
    print("\n" + "-"*80)
    print("Interpretation:")
    print(" - Lower error budget = more reliable circuit")
    print(" - Bell/Grover have moderate error (2-qubit, more gates)")
    print(" - GHZ/W have highest error (3-qubit)")
    print(" - Extra X gates add ~0.04-0.07% error")
    print(" - Error budget correlates with KL-divergence values")
    
except Exception as e:
    print(f"Could not estimate error budget: {e}")

print("\n" + "="*80)


ERROR BUDGET ANALYSIS

This analysis estimates the total error in each circuit
based on the number of gates and measured error rates.

Circuit Error Budget Estimates:

GHZ Circuit (correct):
 Gates: 1 H + 2 CX
 Estimated total error: 3.64%

GHZ Circuit (buggy with X):
 Gates: 1 H + 2 CX + 1 X
 Estimated total error: 3.71%

W State Circuit (correct):
 Gates: 1 RY + 2 X + 1 CRY + 1 CX
 Estimated total error: 3.78%

W State Circuit (buggy with X):
 Gates: 1 RY + 3 X + 1 CRY + 1 CX
 Estimated total error: 3.85%

Bell State Circuit (correct):
 Gates: 1 H + 1 CX
 Estimated total error: 3.08%

Bell State Circuit (buggy with X):
 Gates: 1 H + 1 CX + 1 X
 Estimated total error: 3.18%

Grover Circuit (correct):
 Gates: 8 H + 6 X + 2 CZ
 Estimated total error: 4.29%

Grover Circuit (buggy with X):
 Gates: 8 H + 7 X + 2 CZ
 Estimated total error: 4.38%

--------------------------------------------------------------------------------
Interpretation:
 - Lower error budget = more reliable circuit
 -

In [22]:
print("\n" + "="*80)
print("SIMULATION vs REAL HARDWARE NOISE COMPARISON")
print("="*80)

print(f"\nSimulated Noise Model Parameters:")
print(f" Single-qubit gate error: {error_rate*100:.1f}%")
print(f" Two-qubit gate error: {error_rate*2*100:.1f}%")
print(f" Noise type: Depolarizing")

try:
    # Get average real hardware errors (check first 3 qubits for consistency)
    sq_errors = []
    tq_errors = []
    ro_errors = []
    
    target = backend.target
    
    for qubit in range(min(3, backend.num_qubits)):
        if 'x' in target:
            try:
                sq_errors.append(target['x'][(qubit,)].error)
            except:
                pass
        try:
            ro_errors.append(target['measure'][(qubit,)].error)
        except:
            pass
    
    for i in range(min(2, backend.num_qubits - 1)):
        if (i, i+1) in config.coupling_map:
            try:
                if 'cx' in target:
                    tq_errors.append(target['cx'][(i, i+1)].error)
            except:
                pass
    
    if sq_errors and tq_errors:
        print(f"\nReal Hardware ({backend.name}) Noise:")
        print(f" Single-qubit gate error: {np.mean(sq_errors)*100:.2f}% (avg)")
        print(f" Two-qubit gate error: {np.mean(tq_errors)*100:.2f}% (avg)")
        if ro_errors:
            print(f" Readout error: {np.mean(ro_errors)*100:.2f}% (avg)")
        print(f" Noise type: Mixed (depolarizing + readout + T1/T2)")
        
        print(f"\nNoise Comparison (2-qubit/3-qubit circuits):")
        sim_to_real_sq = error_rate / np.mean(sq_errors)
        sim_to_real_tq = (error_rate * 2) / np.mean(tq_errors)
        
        print(f" Single-qubit: Sim/Real ratio = {sim_to_real_sq:.2f}x")
        print(f" Two-qubit: Sim/Real ratio = {sim_to_real_tq:.2f}x")
        
        if sim_to_real_sq < 1 or sim_to_real_tq < 1:
            print("\n Simulation underestimates real hardware noise.")
            print(" Lower detection ratios on hardware expected.")
        else:
            print("\n Simulation approximates hardware noise reasonably.")
            print(" Detection ratios consistent across platforms.")

except Exception as e:
    print(f"\nCould not compare noise models: {e}")

print("\n" + "="*80)


SIMULATION vs REAL HARDWARE NOISE COMPARISON

Simulated Noise Model Parameters:
 Single-qubit gate error: 2.0%
 Two-qubit gate error: 4.0%
 Noise type: Depolarizing

